### 1. Dataset Exploration

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

relationships = pd.read_csv("train_relationships.csv")
print("Shape:", relationships.shape)
relationships.head()

Shape: (3598, 2)


,p1,p2
0,F0002/MID1,F0002/MID3
1,F0002/MID2,F0002/MID3
2,F0005/MID1,F0005/MID2
3,F0005/MID3,F0005/MID2
4,F0009/MID1,F0009/MID4


In [2]:
#Explore the Folder Structure
#how many unique families are available for training and for generating negative pairs.
import os
from pathlib import Path

train_path = Path("train-faces")
families = os.listdir(train_path)
print("Total Families:", len(families))   #Each folder is one family

Total Families: 786


In [3]:
#Count Total People
people = 0
for family in families:
    people += len(os.listdir(train_path/family))
print("Total People:", people)

Total People: 3965


In [4]:
#Count Images
images = 0
for family in families:
    family_path = train_path / family
    for member in os.listdir(family_path):
        member_path = family_path / member
        images += len(os.listdir(member_path))
print("Total Images:", images)

Total Images: 20080


In [5]:
from pathlib import Path

family = next(Path("train-faces").iterdir())
print(family)
print(len(list(family.iterdir())))

train-faces\F0001
4


In [6]:
print("Train Faces")
print(len(list(Path("train-faces").rglob("*.jpg"))))

Train Faces
20080


In [7]:
relationships.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3598 entries, 0 to 3597
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   p1      3598 non-null   object
 1   p2      3598 non-null   object
dtypes: object(2)
memory usage: 56.3+ KB


In [8]:
#No missing values. so  EDA DONE

In [9]:
from pathlib import Path

print("Original train")
print("Families :", len(list(Path("data/train").iterdir())))
print("People   :", len(list(Path("data/train").rglob("MID*"))))
print("Images   :", len(list(Path("data/train").rglob("*.jpg"))))

print("\nTrain-faces")
print("Families :", len(list(Path("train-faces").iterdir())))
print("People   :", len(list(Path("train-faces").rglob("MID*"))))
print("Images   :", len(list(Path("train-faces").rglob("*.jpg"))))

Original train
Families : 470
People   : 2318
Images   : 12379

Train-faces
Families : 786
People   : 3965
Images   : 20080


In [10]:
orig = set(x.name for x in Path("data/train").iterdir())
faces = set(x.name for x in Path("train-faces").iterdir())

print(len(orig))
print(len(faces))
print(orig == faces)  # Means It is a different training dataset.

470
786
False


### Stage 2 : Data Preparation

In [11]:
# Step 2.1 Build Person → Images Dictionary
# ↓
# Step 2.2 Generate Positive Image Pairs
# ↓
# Step 2.3 Generate Negative Image Pairs
# ↓
# Step 2.4 Create Final Dataset

In [12]:
from pathlib import Path
import os

train_path = Path("train-faces")
person_to_images = {}
for family in os.listdir(train_path):
    family_path = train_path / family
    for person in os.listdir(family_path):
        person_path = family_path / person
        images = []
        for img in os.listdir(person_path):
            images.append(person_path / img)

        # Store only if the person has at least one image
        if images:
            person_to_images[f"{family}/{person}"] = images

In [13]:
#verifying
print("Total Persons:", len(person_to_images))

# 3965 - 3881 = 84 people
# Some MID folders contain no valid images.

Total Persons: 3881


In [14]:
sample_person = next(iter(person_to_images))
print(sample_person)

F0001/MID1


In [15]:
person_to_images[sample_person]

[WindowsPath('train-faces/F0001/MID1/P00001_face2.jpg'),
 WindowsPath('train-faces/F0001/MID1/P00002_face3.jpg'),
 WindowsPath('train-faces/F0001/MID1/P00003_face1.jpg'),
 WindowsPath('train-faces/F0001/MID1/P00004_face3.jpg'),
 WindowsPath('train-faces/F0001/MID1/P00007_face2.jpg'),
 WindowsPath('train-faces/F0001/MID1/P00008_face7.jpg')]

In [16]:
#done verified

In [17]:
#Now will generate positive pairs notice:-
# Yes, positive PERSON pairs are already given.
# No, 
# positive IMAGE pairs are not given.

In [18]:
positive_pairs = []
for _, row in relationships.iterrows():

    person1 = row["p1"]
    person2 = row["p2"]

    if person1 not in person_to_images or person2 not in person_to_images:
        continue

    images1 = person_to_images[person1]
    images2 = person_to_images[person2]

    for img1 in images1:
        for img2 in images2:
            positive_pairs.append((img1, img2, 1))

In [19]:
print("Total Positive Image Pairs:", len(positive_pairs))

Total Positive Image Pairs: 128578


In [20]:
skipped = 0
for _, row in relationships.iterrows():
    if row["p1"] not in person_to_images or row["p2"] not in person_to_images:
        skipped += 1
print("Skipped Relationships:", skipped)
#could not be converted into image pairs because at least one person was missing from the dataset.

Skipped Relationships: 1108


In [21]:
# # so how many relationships actually remained?
# 3881 - 1108 = 3367 relationships were usable.
# we cannot create
# Image A ↔ Image B

In [22]:
positive_pairs[:3]   # Image A ↔ Image B -> label 

[(WindowsPath('train-faces/F0002/MID1/P00009_face3.jpg'),
  WindowsPath('train-faces/F0002/MID3/P00009_face1.jpg'),
  1),
 (WindowsPath('train-faces/F0002/MID1/P00009_face3.jpg'),
  WindowsPath('train-faces/F0002/MID3/P00010_face1.jpg'),
  1),
 (WindowsPath('train-faces/F0002/MID1/P00009_face3.jpg'),
  WindowsPath('train-faces/F0002/MID3/P00011_face3.jpg'),
  1)]

### now negative pairs


In [23]:
# So choose only ONE random image?
# one random to many can generate multiples negative..

In [24]:
import random

negative_pairs = []
used_pairs = set()

all_people = list(person_to_images.keys())
# Keep negatives equal to positives
target = len(positive_pairs)

while len(negative_pairs) < target:

    # Randomly choose two people
    person1 = random.choice(all_people)
    person2 = random.choice(all_people)

    # Same person -> skip
    if person1 == person2:
        continue
    # Different families only
    family1 = person1.split("/")[0]
    family2 = person2.split("/")[0]

    if family1 == family2:
        continue

    # Pick one random image from each person
    img1 = random.choice(person_to_images[person1])
    img2 = random.choice(person_to_images[person2])

    # Treat (A,B) and (B,A) as the same pair
    pair = tuple(sorted([str(img1), str(img2)]))

    if pair in used_pairs:
        continue

    used_pairs.add(pair)
    negative_pairs.append((img1, img2, 0))
    
print("Total Negative Image Pairs:", len(negative_pairs))

Total Negative Image Pairs: 128578


In [25]:
print("Positive:", len(positive_pairs))
print("Negative:", len(negative_pairs))
print("Total Dataset:", len(positive_pairs) + len(negative_pairs))

Positive: 128578
Negative: 128578
Total Dataset: 257156


In [26]:
# Dataset is Balanced

In [27]:
#verifying
negative_pairs[:2]

[(WindowsPath('train-faces/F0749/MID2/P07845_face1.jpg'),
  WindowsPath('train-faces/F0430/MID1/P04523_face1.jpg'),
  0),
 (WindowsPath('train-faces/F0113/MID5/P01178_face2.jpg'),
  WindowsPath('train-faces/F0882/MID5/P12356_face2.jpg'),
  0)]

In [28]:
img1, img2, label = negative_pairs[0]
print(img1)
print(img2)
print(label)

train-faces\F0749\MID2\P07845_face1.jpg
train-faces\F0430\MID1\P04523_face1.jpg
0


In [29]:
print(len(negative_pairs))
print(len(used_pairs))  # ✅ No duplicate image pairs.

128578
128578


In [30]:
# so will do combine and suffle suflle means mixture of both 0 & 1

In [31]:
# Combine positive and negative image pairs
all_pairs = positive_pairs + negative_pairs
print("Total Training Samples:", len(all_pairs))

Total Training Samples: 257156


In [32]:
import random
random.shuffle(all_pairs)

In [33]:
print(all_pairs[:10])

[(WindowsPath('train-faces/F0754/MID1/P07958_face1.jpg'), WindowsPath('train-faces/F0754/MID2/P07944_face3.jpg'), 1), (WindowsPath('train-faces/F0226/MID3/P02410_face3.jpg'), WindowsPath('train-faces/F0030/MID2/P00300_face1.jpg'), 0), (WindowsPath('train-faces/F0437/MID1/P04608_face2.jpg'), WindowsPath('train-faces/F0816/MID2/P08631_face1.jpg'), 0), (WindowsPath('train-faces/F0742/MID1/P07775_face4.jpg'), WindowsPath('train-faces/F0599/MID3/P10738_face1.jpg'), 0), (WindowsPath('train-faces/F0038/MID1/P00398_face1.jpg'), WindowsPath('train-faces/F0677/MID1/P07066_face1.jpg'), 0), (WindowsPath('train-faces/F0601/MID5/P11973_face2.jpg'), WindowsPath('train-faces/F0601/MID9/P06242_face1.jpg'), 1), (WindowsPath('train-faces/F0335/MID3/P11764_face1.jpg'), WindowsPath('train-faces/F0143/MID2/P01509_face3.jpg'), 0), (WindowsPath('train-faces/F0796/MID5/P08421_face5.jpg'), WindowsPath('train-faces/F0997/MID4/P10531_face3.jpg'), 0), (WindowsPath('train-faces/F0642/MID1/P06736_face1.jpg'), Window

In [34]:
for pair in all_pairs[:10]:
    print(pair[2])

1
0
0
0
0
1
0
0
1
1


In [35]:
from sklearn.model_selection import train_test_split
train_pairs, val_pairs = train_test_split(
    all_pairs,
    test_size=0.2,
    random_state=42,
    shuffle=True
)
print("Training Samples:", len(train_pairs))
print("Validation Samples:", len(val_pairs))

Training Samples: 205724
Validation Samples: 51432


In [36]:
train_pos = sum(label for _, _, label in train_pairs)
train_neg = len(train_pairs) - train_pos

val_pos = sum(label for _, _, label in val_pairs)
val_neg = len(val_pairs) - val_pos

print("\nTraining Set")    
print("Positive:", train_pos)
print("Negative:", train_neg)

print("\nValidation Set")
print("Positive:", val_pos)
print("Negative:", val_neg)


Training Set
Positive: 102957
Negative: 102767

Validation Set
Positive: 25621
Negative: 25811


In [37]:
# Difference is only 144 out of 205k samples (0.07%).
# Excellent.

In [38]:
from torch.utils.data import Dataset
from PIL import Image

class KinshipDataset(Dataset):

    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        img1_path, img2_path, label = self.pairs[idx]

        img1 = Image.open(img1_path).convert("RGB")
        img2 = Image.open(img2_path).convert("RGB")

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, label

In [39]:
# (Path,Path,Label) to (Image Tensor,Image Tensor,Label)

In [40]:
# Stage 3 (continued): Image Transform

# Now we tell PyTorch how every image should be prepared before entering the CNN

from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),  #CNN cannot train on different-sized images. that's why resize
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
]) 

In [41]:
# Stage 4 : Create Dataset Objects
train_dataset = KinshipDataset(train_pairs, transform=transform)
val_dataset = KinshipDataset(val_pairs, transform=transform)

In [42]:
#verifying 
print(len(train_dataset))
print(len(val_dataset))

205724
51432


In [43]:
#teesting on one sample
img1, img2, label = train_dataset[0]
print(img1.shape)
print(img2.shape)
print(label)                      # Stage 4 ✔ Image Transform

torch.Size([3, 224, 224])
torch.Size([3, 224, 224])
0


### DataLoader

In [44]:
from torch.utils.data import DataLoader
#Train Loader
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)
#validation loader
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)
# Why shuffle=True only for training?
# Model should see data in different order every epoch.

# Validation 
# We only measure accuracy.
# Order doesn't matter.

In [45]:
print(len(train_loader))  #205724/64=3215
print(len(val_loader))    #51432/64=804

3215
804


In [46]:
# testing one batch
img1, img2, label = next(iter(train_loader))
print(img1.shape)
print(img2.shape)
print(label.shape)

torch.Size([64, 3, 224, 224])
torch.Size([64, 3, 224, 224])
torch.Size([64])


In [47]:
# What is an Embedding? This is one of the most important concepts in Deep Learning.
# Suppose the original image is 224 × 224 × 3 = 150,528 numbers
# This is huge.
# CNN compresses it into something like 512 numbers
# called
# Face Embedding

In [48]:
# Stage 7.1 : Import the Pretrained Model :ResNet18 
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.9.1+cu128
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [49]:
# Next step: Stage 7.2 – Load ResNet18
# We'll use a pretrained ResNet18 and remove its classification head so it outputs a 512-dimensional
# face embedding.
from torchvision import models
import torch.nn as nn

# Load pretrained ResNet18
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Remove the final classification layer
resnet.fc = nn.Identity()

#print(resnet)

In [50]:
dummy = torch.randn(1, 3, 224, 224).cuda()
resnet = resnet.cuda()
with torch.no_grad():
    embedding = resnet(dummy)
print(embedding.shape)

torch.Size([1, 512])


In [51]:
# These 512 numbers are the embedding (feature vector) representing the face.
import torch
import torch.nn as nn
from torchvision import models

class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        # Backbone
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Remove classifier
        self.backbone.fc = nn.Identity()
        # Similarity classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            #nn.Sigmoid()
            #criterion = nn.BCELoss() #bz:-BCEWithLogitsLoss internally applies the sigmoid in a numerically stable way.
                                     # It is the PyTorch-recommended approach for binary classification.
        )
    def forward_once(self, x):
        return self.backbone(x)
    def forward(self, img1, img2):      #Why forward_once()?
        emb1 = self.forward_once(img1)  #because both images must pass through the exact same ResNet18 (shared weights).
        emb2 = self.forward_once(img2)

        diff = torch.abs(emb1 - emb2)  #|EmbeddingA - EmbeddingB|
        output = self.classifier(diff)
        return output

In [52]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SiameseNetwork().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)
EPOCHS = 10

In [53]:
 #verifying 
img1 = torch.randn(4,3,224,224).to(device)
img2 = torch.randn(4,3,224,224).to(device)
output = model(img1,img2)

print(output)

tensor([[0.1163],
        [0.2220],
        [0.3046],
        [0.0295]], device='cuda:0', grad_fn=<AddmmBackward0>)


### Stage 9: Training Setup,

In [54]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [55]:
print(torch.cuda.memory_allocated() / 1024**2, "MB")

268.7890625 MB


In [56]:
#Step 1 : Validation Function
import torch
def validate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for img1, img2, labels in dataloader:

            img1 = img1.to(device)
            img2 = img2.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(img1, img2)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            predictions = (torch.sigmoid(outputs) > 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / len(dataloader)
    accuracy = correct / total

    return avg_loss, accuracy

In [57]:
#Step 2 : Create checkpoint folder
import os

os.makedirs("checkpoints_train_faces", exist_ok=True)

In [58]:
start_epoch = 0
best_acc = 0.0

if os.path.exists("checkpoints_train_faces/latest.pth"):

    checkpoint = torch.load(
        "checkpoints_train_faces/latest.pth",
        map_location=device
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_acc = checkpoint["best_acc"]

    print(f"Resuming from Epoch {start_epoch}")

else:
    print("Starting Fresh Training")

Resuming from Epoch 8


In [59]:
from torch.cuda.amp import GradScaler, autocast
scaler = GradScaler()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

C:\Users\rishu\AppData\Local\Temp\ipykernel_1280\434963958.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [60]:
from tqdm import tqdm
import torch

train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(start_epoch, EPOCHS):

    model.train()
    running_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for img1, img2, labels in progress_bar:

        img1 = img1.to(device, non_blocking=True)
        img2 = img2.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            outputs = model(img1, img2)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.1e}"
        )

    train_loss = running_loss / len(train_loader)

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_acc)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_acc": best_acc,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "val_accuracies": val_accuracies,
        }, "checkpoints_train_faces/best_model.pth")

        print("✅ Best Model Saved")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_acc": best_acc,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies,
    }, "checkpoints_train_faces/latest.pth")

    print("-"*60)
    print(f"Epoch      : {epoch+1}/{EPOCHS}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc*100:.2f}%")
    print(f"Best Acc   : {best_acc*100:.2f}%")
    print("💾 Latest Checkpoint Saved\n")

Epoch 9/10:   0%|          | 0/3215 [00:00<?, ?it/s]C:\Users\rishu\AppData\Local\Temp\ipykernel_1280\920710261.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 9/10: 100%|██████████| 3215/3215 [16:00<00:00,  3.35it/s, loss=0.0297, lr=1.0e-04]


✅ Best Model Saved
------------------------------------------------------------
Epoch      : 9/10
Train Loss : 0.0613
Val Loss   : 0.0959
Val Acc    : 96.37%
Best Acc   : 96.37%
💾 Latest Checkpoint Saved



Epoch 10/10: 100%|██████████| 3215/3215 [16:09<00:00,  3.32it/s, loss=0.0279, lr=1.0e-04]


✅ Best Model Saved
------------------------------------------------------------
Epoch      : 10/10
Train Loss : 0.0379
Val Loss   : 0.0831
Val Acc    : 96.94%
Best Acc   : 96.94%
💾 Latest Checkpoint Saved



### Evaluating

In [61]:
# 1. Imports
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

In [62]:
# 2. Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [63]:
# 3. Image Transform
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [64]:
# 4. Read CSV & Create Pairs
all_pairs
train_pairs, val_pairs = train_test_split(
    all_pairs,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [65]:
# Dataset
train_dataset = KinshipDataset(train_pairs, transform)
val_dataset = KinshipDataset(val_pairs, transform)

In [75]:
# Load Saved Model

checkpoint = torch.load(
    "checkpoints_train_faces/best_model.pth",
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model Loaded Successfully")

Model Loaded Successfully


In [76]:
#9. Evaluate on Validation Set
all_labels = []
all_preds = []
all_probs = []

from tqdm import tqdm

with torch.no_grad():
    for img1, img2, labels in tqdm(val_loader):
        img1 = img1.to(device)
        img2 = img2.to(device)

        outputs = model(img1, img2)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())    

100%|██████████| 804/804 [03:07<00:00,  4.29it/s]


In [77]:
all_labels = np.array(all_labels)
all_preds = np.array(all_preds).flatten()
all_probs = np.array(all_probs).flatten()

print("Accuracy :", accuracy_score(all_labels, all_preds))
print("Precision:", precision_score(all_labels, all_preds))
print("Recall   :", recall_score(all_labels, all_preds))
print("F1 Score :", f1_score(all_labels, all_preds))
print("ROC AUC  :", roc_auc_score(all_labels, all_probs))

print("\nConfusion Matrix")
print(confusion_matrix(all_labels, all_preds))

Accuracy : 0.96935759838233
Precision: 0.9833943146636646
Recall   : 0.9546075484953749
F1 Score : 0.9687871345955795
ROC AUC  : 0.9965214518835751

Confusion Matrix
[[25398   413]
 [ 1163 24458]]


In [78]:
# sample_submission
submission = pd.read_csv("sample_submission.csv")
submission.head(2)


,img_pair,is_related
0,face05508.jpg-face01210.jpg,0
1,face05750.jpg-face00898.jpg,0


In [79]:
predictions = []
model.eval()

with torch.no_grad():
    for pair in tqdm(submission["img_pair"]):

        img1_name, img2_name = pair.split("-")

        img1 = Image.open(os.path.join("data/test", img1_name)).convert("RGB")
        img2 = Image.open(os.path.join("data/test", img2_name)).convert("RGB")

        img1 = transform(img1).unsqueeze(0).to(device)
        img2 = transform(img2).unsqueeze(0).to(device)

        output = model(img1, img2)
        prob = torch.sigmoid(output).item()
        predictions.append(prob)

100%|██████████| 5310/5310 [01:03<00:00, 83.25it/s]


In [80]:
submission = pd.read_csv("sample_submission.csv")
# Fill predictions
submission["is_related"] = predictions
# Save as a NEW file
submission.to_csv("submission_train-faces_resnet18.csv", index=False)
print("✅ Saved as submission_train-faces_resnet18.csv")

✅ Saved as submission_train-faces_resnet18.csv
